# TextVQA: Gemma 3 12B Base vs V5 LoRA

Independent **no-retraining** evaluation of scene-text visual question answering, requested for the URAP handoff. This uses the official **TextVQA v0.5.1 validation split** (5,000 labeled questions), **official VQA soft-answer accuracy**, and **image-clustered paired significance testing**. It does not claim test-set leaderboard performance.

**Prerequisites:** Google Colab with an A100 GPU, a Hugging Face account that has accepted the Gemma license, an `HF_TOKEN` in Colab Secrets, and Drive access to `MyDrive/IAT/models/gemma3-12b-v5-v4-optimized-freshbase/`. Expect an ~7.1 GB official image download into temporary Colab storage, a 12B model download and a potentially long 2-model GPU evaluation. GPU time/Colab credits may be charged.

No training. No model uploads. No changes to the production or research datasets. The saved V5 adapter is applied to the **same** original Gemma 3 base weights used for the baseline. Only raw predictions and summaries are written to your Drive results folder.

For a quick smoke test, set `N_QUESTIONS = 20` below, run, then switch to `5000` and rerun. The runner skips previously completed predictions. Results for a subset are labeled **pilot**, not full-validation results.

Official references: [TextVQA](https://textvqa.org/download/) · [Meta TextVQA dataset](https://huggingface.co/datasets/facebook/textvqa) · [Meta MMF evaluator](https://github.com/facebookresearch/mmf/blob/main/mmf/utils/m4c_evaluators.py) · [Gemma-3-12b-it](https://huggingface.co/google/gemma-3-12b-it).

In [ ]:
# Colab GPU runtime: Runtime -> Change runtime type -> A100 GPU.
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=5.10.1', 'peft>=0.19.0', 'accelerate',
    'bitsandbytes', 'Pillow', 'numpy', 'huggingface_hub'])
print('Dependencies installed. If imports complain after upgrading, restart runtime and rerun.')

In [ ]:
from pathlib import Path
import torch
from google.colab import drive, userdata
from huggingface_hub import login
assert torch.cuda.is_available(), 'Select a GPU runtime first'
print('GPU:', torch.cuda.get_device_name(0))
drive.mount('/content/drive')
token = userdata.get('HF_TOKEN')
assert token, 'Please add HF_TOKEN in Colab Secrets and enable notebook access'
login(token=token, add_to_git_credential=False)
adapter = Path('/content/drive/MyDrive/IAT/models/gemma3-12b-v5-v4-optimized-freshbase')
assert (adapter / 'adapter_model.safetensors').is_file(), f'Missing saved V5 adapter: {adapter}'
print('V5 adapter verified:', adapter)

In [ ]:
# Retrieve the exact reviewed source version, so later GitHub edits do not change this run.
import urllib.request
SCRIPT_COMMIT = 'cb0a4b64d6c409cf16cf922eefbc3c32217a4d44'
script_url = f'https://raw.githubusercontent.com/alicasuu/gemma3-bias-finetuning/{SCRIPT_COMMIT}/scripts/textvqa_compare.py'
script_path = Path('/content/textvqa_compare.py')
script_path.write_bytes(urllib.request.urlopen(script_url, timeout=60).read())
print('Downloaded versioned runner:', SCRIPT_COMMIT)

In [ ]:
# 5000 = complete validation; use 20 or 200 temporarily for a smoke test.
# Each completed answer is durably saved in Drive, so this cell is resumable.
N_QUESTIONS = 5000
OUTDIR = '/content/drive/MyDrive/IAT/results/gemma/textvqa_base_vs_v5'
subprocess.check_call([sys.executable, str(script_path),
    '--n', str(N_QUESTIONS), '--adapter', str(adapter), '--outdir', OUTDIR])

In [ ]:
# The summary is generated only after every Base and V5 answer is present.
import json
from IPython.display import display, Markdown
summary_file = Path(OUTDIR) / f'summary_n{N_QUESTIONS}.json'
summary = json.loads(summary_file.read_text())
lines = [
    f"**Base:** {summary['base_vqa_accuracy_percent']:.2f}%",
    f"**V5:** {summary['v5_vqa_accuracy_percent']:.2f}%",
    f"**Delta (V5 - Base):** {summary['delta_percentage_points']:+.2f} percentage points",
    f"**95% CI:** [{summary['delta_95pct_cluster_bootstrap_ci_pp'][0]:+.2f}, {summary['delta_95pct_cluster_bootstrap_ci_pp'][1]:+.2f}] pp",
    f"**Paired p-value:** {summary['paired_image_cluster_signflip_two_sided_p']:.4g}",
    f"**Questions:** {summary['n_questions']}",
]
display(Markdown('  \n'.join(lines)))
print('Raw results:', Path(OUTDIR) / 'base_raw.csv', Path(OUTDIR) / 'v5_raw.csv')
print('Full report:', summary_file)

## How to report this to Debbie

- **Method:** Same TextVQA v0.5.1 validation questions/images, Gemma-3-12B-IT before and after V5 LoRA, greedy decoding and the same prompt. No fine-tuning on TextVQA.
- **Results:** Report Base and V5 official soft VQA accuracy (%), V5–Base percentage-point difference, clustered 95% confidence interval, paired two-sided permutation p-value, number of questions and unique images.
- **Interpretation:** Positive significant delta = evidence V5 improves this benchmark; negative significant delta = evidence of deterioration; non-significant result = no conclusive difference. Do not treat `p > 0.05` as proof of equivalence.
- **Scope:** TextVQA primarily tests reading text in images; it is one aspect of visual-language ability, not a complete measure of all VLM capabilities.

For the 3-slide pitch: (1) debiasing objective and Base→V5 method; (2) previous bias metrics; (3) TextVQA ability-retention comparison + p-value and next steps. Add numerical findings only after the full run completes.